# 06 · 张量自动求导（二）：matmul 与更多算子

> **本节属于 Part 3 · 张量自动求导。**

上一节我们搭出了 `Tensor` 的地基（逐元素 + 广播反向）。本节补齐神经网络真正需要的几类算子，并逐一用 `gradcheck` 验收：

- **矩阵乘法 `matmul`**（全连接层的核心）
- **归约** `sum / mean(axis)`
- **激活** `relu / sigmoid / tanh / exp / log`
- **形状变换** `reshape / transpose / getitem`

从本节起，我们直接用包里已经写好的完整 `Tensor`（`from minitorch import Tensor`），并用 `inspect.getsource` 把它的真实实现亮出来，确保你看到的就是包里运行的代码。

## 学习目标

- 掌握 **matmul 的反向公式** $dA = dC\,B^\top,\quad dB = A^\top dC$ 并验证
- 理解归约/激活/形状变换各自的反向规则
- 学会用 `gradcheck`（数值梯度）+ PyTorch 双重验收任意算子

In [ ]:
import inspect
import numpy as np
from minitorch import Tensor, numerical_gradient, rel_error
import torch

## 1. 矩阵乘法 matmul

矩阵乘法是全连接层 $O = XW$ 的核心。它的反向公式非常优雅：若 $C = AB$，上游梯度为 $dC$，则

$$dA = dC\,B^\top, \qquad dB = A^\top dC$$

（直觉：要凑出形状——$dA$ 必须和 $A$ 同形，唯一的凑法就是 $dC\,B^\top$。）下面是包里的真实实现：

In [ ]:
print(inspect.getsource(Tensor.matmul))

In [ ]:
# 验证：数值梯度 + PyTorch
A = np.random.randn(5, 3); B = np.random.randn(3, 4)
tA, tB = Tensor(A), Tensor(B)
(tA @ tB).sum().backward()

gA = numerical_gradient(lambda x: (x @ B).sum(), A.copy())
gB = numerical_gradient(lambda x: (A @ x).sum(), B.copy())
print("matmul dA 相对误差:", rel_error(tA.grad, gA))
print("matmul dB 相对误差:", rel_error(tB.grad, gB))

## 2. 归约：sum / mean

损失通常是一个标量，所以几乎每个网络最后都要 `sum` 或 `mean`。反向时，标量的梯度要**广播回**原来的形状（`mean` 还要再除以元素个数）。

In [ ]:
x = np.random.randn(4, 6)
tx = Tensor(x); tx.mean(axis=1).sum().backward()
g = numerical_gradient(lambda v: v.mean(axis=1).sum(), x.copy())
print("mean(axis=1) 相对误差:", rel_error(tx.grad, g))

## 3. 激活函数

非线性是神经网络的灵魂。这些激活的反向都是逐元素的，复习一下导数：

| 激活 | 前向 | 反向（局部导数）|
|---|---|---|
| relu | $\max(0,x)$ | $\mathbb{1}[x>0]$ |
| sigmoid | $\sigma(x)$ | $\sigma(1-\sigma)$ |
| tanh | $\tanh(x)$ | $1-\tanh^2$ |
| exp | $e^x$ | $e^x$ |
| log | $\ln x$ | $1/x$ |

In [ ]:
x = np.random.randn(3, 5)
checks = {
    "relu":    (lambda v: np.maximum(0, v),       lambda t: t.relu()),
    "sigmoid": (lambda v: 1/(1+np.exp(-v)),        lambda t: t.sigmoid()),
    "tanh":    (np.tanh,                           lambda t: t.tanh()),
    "exp":     (np.exp,                            lambda t: t.exp()),
}
for name, (np_fn, t_fn) in checks.items():
    tx = Tensor(x); t_fn(tx).sum().backward()
    g = numerical_gradient(lambda v: np_fn(v).sum(), x.copy())
    print(f"  {name:8s} 相对误差: {rel_error(tx.grad, g):.2e}")

xp = np.abs(np.random.randn(3, 5)) + 0.3      # log 需要正数
tx = Tensor(xp); tx.log().sum().backward()
print(f"  {'log':8s} 相对误差: {rel_error(tx.grad, numerical_gradient(lambda v: np.log(v).sum(), xp.copy())):.2e}")

## 4. 形状变换：reshape / transpose / getitem

这些"不改数值、只改排列"的操作，反向同样要把梯度按相同的方式搬回去：reshape 反向是 reshape 回去；transpose 反向是逆置换；getitem（切片/索引）反向是把梯度**散射**回被取的位置。

In [ ]:
x = np.random.randn(2, 6)
tx = Tensor(x)
(tx.reshape(3, 4).transpose()[1]).sum().backward()    # reshape -> 转置 -> 取一行
g = numerical_gradient(lambda v: v.reshape(3, 4).transpose()[1].sum(), x.copy())
print("reshape+transpose+getitem 相对误差:", rel_error(tx.grad, g))

## 综合演练：一个两层网络的前向 + 自动反向

现在把它们组合起来——这正是 nb02 我们**手推**过的两层网络，但这次梯度**全自动**，且向量化处理整批数据。

In [ ]:
from minitorch import set_seed
set_seed(0)
N = 16
X = np.random.randn(N, 3)
Y = np.random.randn(N, 1)
W1 = Tensor(np.random.randn(3, 8) * 0.5)
W2 = Tensor(np.random.randn(8, 1) * 0.5)

# 前向：tanh(X W1) W2，再算 MSE
H = (Tensor(X) @ W1).tanh()
O = H @ W2
diff = O - Tensor(Y)
loss = (diff * diff).mean()
loss.backward()

# 验证 W1 的梯度
def loss_np(W1v):
    O = np.tanh(X @ W1v) @ W2.data
    return ((O - Y) ** 2).mean()
print("loss =", float(loss.data))
print("W1.grad 相对误差:", rel_error(W1.grad, numerical_gradient(loss_np, W1.data.copy())))

# 与 PyTorch 对照
Xt, Yt = torch.tensor(X), torch.tensor(Y)
W1t = torch.tensor(W1.data, requires_grad=True)
W2t = torch.tensor(W2.data, requires_grad=True)
((torch.tanh(Xt @ W1t) @ W2t - Yt) ** 2).mean().backward()
print("W1.grad vs PyTorch:", rel_error(W1.grad, W1t.grad.numpy()))

## 📦 沉淀进 minitorch

以上算子都已在 **`minitorch/tensor.py`** 中实现，并通过 `tests/test_tensor.py` 的全部梯度检查。看看 `Tensor` 现在拥有的"武器库"：

In [ ]:
print("Tensor 公开方法:", [m for m in dir(Tensor) if not m.startswith("_")])

## 小练习

1. **批量 matmul**：构造 `A:(10,5,3)`、`B:(3,4)`，验证 `A @ B` 的反向（提示：包里的 `_unbroadcast` 会自动处理 batch 维）。
2. **softmax 预热**：用现有算子（`exp`、`sum`、`/`）拼出 `softmax(x) = exp(x)/sum(exp(x))`，并用 `gradcheck` 验证——下一个 Part 我们会用它做分类。
3. **数值稳定性**：直接对很大的输入做 `exp` 会溢出。想想怎么先减去最大值再 exp（这是 softmax 的标准技巧，Part 4 会正式实现）。

## 小结 & 下一站

✅ 我们补齐了 matmul、归约、激活、形状变换，并逐一通过数值梯度检查与 PyTorch 对照。`Tensor` 现在已经是一个**功能完整**的张量 autograd 引擎。

**下一站 → `07_topo_backward_and_no_grad`**：我们把反向引擎本身讲透（拓扑排序、梯度累加），加上 `no_grad` 与 `detach`，并用这套完整引擎**端到端训练一个网络**——为 Part 4 把这些样板封装成 `nn.Module` 做铺垫。